In [17]:
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from google.colab import files

print("Upload your accident dataset CSV files")
uploaded = files.upload()

# Combine uploaded files
dfs = []
for file in uploaded.keys():
    df_temp = pd.read_csv(file)
    dfs.append(df_temp)
df = pd.concat(dfs, ignore_index=True)

# --- THE FIX: DATA CLEANING ---
# 1. Remove the country-wide "Total" row that ruins the math
df = df[df['Location'].str.strip().str.lower() != 'total']

# 2. Clean up the Location names (remove accidental quotes or spaces from the CSV)
df['Location'] = df['Location'].str.replace('"', '').str.strip()


# Preprocessing: Now the max() will correctly be Colombo (~21,000) instead of the Grand Total
df['accident_percentage'] = (df['Total'] / df['Total'].max()) * 100

def risk_level(percent):
    # Adjusted Thresholds to be more realistic for Sri Lankan traffic
    if percent >= 45:
        return 2   # High (e.g., Colombo, Gampaha)
    elif percent >= 15:
        return 1   # Medium (e.g., Kurunegala, Kandy, Galle)
    else:
        return 0   # Low (Smaller towns and villages)

df['risk'] = df['accident_percentage'].apply(risk_level)

# Prepare Training Data
X = df.drop(columns=['Location', 'Total', 'accident_percentage', 'risk'])
y = df['risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"✅ Model Trained Successfully. Accuracy: {accuracy_score(y_test, y_pred):.2f}")

Upload your accident dataset CSV files


Saving road_accident_data_by_vehicle_type_2012.csv to road_accident_data_by_vehicle_type_2012 (1).csv
Saving road_accident_data_by_vehicle_type_2010.csv to road_accident_data_by_vehicle_type_2010 (1).csv
Saving road_accident_data_by_vehicle_type_2011.csv to road_accident_data_by_vehicle_type_2011 (1).csv
✅ Model Trained Successfully. Accuracy: 0.87


In [18]:
API_KEY = "8da5c6746a8e0067161af359fc276a83"

def get_sri_lanka_coordinates(city_input):
    # The ',LK' forces the API to only look inside Sri Lanka
    geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city_input},LK&limit=1&appid={API_KEY}"
    try:
        res = requests.get(geo_url).json()
        if len(res) == 0:
            print(f"❌ Error: Could not locate '{city_input}' in Sri Lanka.")
            return None, None, None

        lat, lon = res[0]['lat'], res[0]['lon']
        official_name = res[0]['name']
        return lat, lon, official_name
    except Exception as e:
        print(f"API Error: {e}")
        return None, None, None

def get_weather(lat, lon):
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    res = requests.get(url).json()

    if 'main' not in res:
        return 28, 70, 0  # fallback values

    temp = res['main']['temp']
    humidity = res['main']['humidity']
    rainfall = res.get('rain', {}).get('1h', 0)

    return temp, humidity, rainfall

In [44]:
def get_city_traffic_data(city_name):
    """
    Searches the dataset for the specific city and returns its traffic features.
    """
    # Search the 'Location' column for the city name (ignoring uppercase/lowercase)
    match = df[df['Location'].str.contains(city_name, case=False, na=False)]

    if not match.empty:
        # If found (like Gampaha or Kurunegala), return its actual average traffic data
        return match[X.columns].mean().tolist()
    else:
        # If it's a small village not in the police records (like Hungama or Ranna)
        # We use the national median as a safe baseline for the AI
        return X.median().tolist()

def get_elevation_and_terrain(lat, lon):
    """
    Uses a free topographic API to find the elevation and dynamically classify the terrain.
    """
    # Open-Meteo Elevation API (No API Key Required!)
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"

    try:
        res = requests.get(url).json()
        elevation = res['elevation'][0] # Height in meters

        # Dynamic Geographical Logic based on altitude
        if elevation > 400:
            return "hill_country", elevation
        elif elevation < 30:
            return "coastal_or_lowland", elevation
        else:
            return "flat_open", elevation

    except Exception as e:
        print(f"Elevation API Error: {e}")
        return "general", 0

def final_risk_entire_country(city_input):
    # 1. Get Coordinates & Weather
    lat, lon, official_name = get_sri_lanka_coordinates(city_input)
    if lat is None: return {"error": "Location not found."}
    temp, humidity, rainfall = get_weather(lat, lon)

    # 2. Get Historical Traffic Risk
    actual_city_data = get_city_traffic_data(city_input)
    data_df = pd.DataFrame([actual_city_data], columns=X.columns)
    base_risk = model.predict(data_df)[0]

    if base_risk == 2: base_risk_text = "High"
    elif base_risk == 1: base_risk_text = "Medium"
    else: base_risk_text = "Low"

    # 3. GET DYNAMIC TERRAIN VIA SATELLITE API
    terrain_type, elevation = get_elevation_and_terrain(lat, lon)

    # 4. Calculate Final Risk
    if rainfall > 5 and base_risk_text == "High": final = "🚨 VERY HIGH RISK"
    elif rainfall > 0 and base_risk_text == "High": final = "🚨 HIGH RISK"
    elif rainfall > 5: final = "⚠️ HIGH RISK"
    elif rainfall > 0: final = "🟡 MEDIUM RISK"
    elif base_risk_text == "High": final = "⚠️ HIGH RISK"
    elif base_risk_text == "Medium": final = "🟡 MEDIUM RISK"
    else: final = "🟢 LOW RISK"

    # ---------------------------------------------------------
    # 5. GENERATE DYNAMIC SAFETY ADVICE (FIXED!)
    # ---------------------------------------------------------
    safety_tips = []

    # Tip A: Based on Dynamic Elevation
    if terrain_type == "hill_country":
        safety_tips.append(f"⛰️ Terrain ({elevation}m): Steep slopes and sharp bends. Use lower gears when going downhill.")
    elif terrain_type == "coastal_or_lowland":
        safety_tips.append(f"🌊 Terrain ({elevation}m): Lowland/Coastal area. Be prepared for crosswinds or heavy urban traffic.")
    else:
        safety_tips.append(f"🛣️ Terrain ({elevation}m): Standard elevation. Be highly alert for stray animals crossing the road.")

    # Tip B: Based on Historical Accidents (Restored Medium and Low!)
    if base_risk_text == "High":
        safety_tips.append("🛑 History: High accident rate area. Maintain strict speed limits (max 40km/h).")
    elif base_risk_text == "Medium":
        safety_tips.append("⚠️ History: Moderate traffic zone. Keep a safe following distance and watch for pedestrians.")
    else:
        safety_tips.append("✅ History: Historically safe traffic zone. Standard riding rules apply.")

    # Tip C: Based on Live Weather
    if rainfall > 5:
        safety_tips.append("🌧️ Weather: Heavy Rain. Visibility is poor. Pull over safely if possible.")
    elif rainfall > 0:
        safety_tips.append("☔ Weather: Slippery Roads. Avoid sudden braking and sharp turns.")

    final_advice = " | ".join(safety_tips)

    return {
        "searched_input": city_input,
        "official_location": f"{official_name}, Sri Lanka",
        "elevation": f"{elevation} meters",
        "temperature": f"{temp}°C",
        "rainfall": f"{rainfall} mm/h",
        "base_historical_risk": base_risk_text,
        "final_risk_level": final,
        "safety_advice": final_advice
    }

In [46]:
print("--- Test 1: High Risk City ---")
print(final_risk_entire_country("kurunagala"))

print("\n--- Test 2: Another High Risk City ---")
print(final_risk_entire_country("gampaha"))

print("\n--- Test 3: Small Village (Not in dataset) ---")
print(final_risk_entire_country("colombo"))

--- Test 1: High Risk City ---
{'searched_input': 'kurunagala', 'official_location': 'Kurunegala, Sri Lanka', 'elevation': '121.0 meters', 'temperature': '24.94°C', 'rainfall': '0 mm/h', 'base_historical_risk': 'Low', 'final_risk_level': '🟢 LOW RISK', 'safety_advice': '🛣️ Terrain (121.0m): Standard elevation. Be highly alert for stray animals crossing the road. | ✅ History: Historically safe traffic zone. Standard riding rules apply.'}

--- Test 2: Another High Risk City ---
{'searched_input': 'gampaha', 'official_location': 'Gampaha, Sri Lanka', 'elevation': '18.0 meters', 'temperature': '29.95°C', 'rainfall': '0 mm/h', 'base_historical_risk': 'High', 'final_risk_level': '⚠️ HIGH RISK', 'safety_advice': '🌊 Terrain (18.0m): Lowland/Coastal area. Be prepared for crosswinds or heavy urban traffic. | 🛑 History: High accident rate area. Maintain strict speed limits (max 40km/h).'}

--- Test 3: Small Village (Not in dataset) ---
{'searched_input': 'colombo', 'official_location': 'Colombo, S